<a href="https://colab.research.google.com/github/mimiktam/BU_AI_660_Assignments/blob/main/MS_AI_660_Week3_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile week3_assignment.py

########################################################
# Retail_pipeline.py – contains the RetailPipeline class (the core logic)
########################################################
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from kneed import KneeLocator

class RetailPipeline:
    def __init__(self, data_path):
        self.df = pd.read_csv(data_path, encoding='ISO-8859-1')
        print(self.df.columns)
        self.cleaned_df = None
        self.snapshot_date = None
        self.rfm_df = None
        self.model = None

    # Cleaning
    def phase_0_prep(self):
        """Clean data, remove noise, and set snapshot date."""
        df = self.df.dropna(subset=['Customer ID']).copy()
        df = df[~df['Invoice'].astype(str).str.startswith('C')]
        df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]
        df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

        # Consistent snapshot date for the whole pipeline
        self.snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
        self.cleaned_df = df
        print("✅ Phase 0: Data cleaning complete.")
        return self.cleaned_df

    # Feature Engineering
    def phase_1_rfm(self):
        """Create RFM metrics."""
        self.cleaned_df['TotalSum'] = self.cleaned_df['Quantity'] * self.cleaned_df['Price']
        self.rfm_df = self.cleaned_df.groupby('Customer ID').agg({
            'InvoiceDate': lambda x: (self.snapshot_date - x.max()).days,
            'Invoice': 'nunique',
            'TotalSum': 'sum'
        }).rename(columns={'InvoiceDate': 'Recency', 'Invoice': 'Frequency', 'TotalSum': 'Monetary'})
        print("✅ Phase 1: RFM metrics calculated.")
        return self.rfm_df

    # Intelligence – takes the RFM numbers calculated (Recency, Frequency, Monetary) & transforms
    #                              them into actionable customer groups
    #
    # Scaling: It uses StandardScaler. This is crucial because Recency (days) and Monetary (dollars)
    # # are on totally different scales. Scaling ensures the algorithm treats them as equally important.
    #
    # Automatic Optimization: It runs the model 10 times (with 1 to 10 clusters) and uses
    # KneeLocator # to mathematically pick the best number of clusters, preventing you from having
    # to guess.
    #
    # Labeling: It returns an array of "labels" (e.g., [0, 2, 1, 0...]). Each number corresponds to a
    # specific customer cluster (e.g., "VIPs," "At-Risk," "Newbies").
    def phase_2_cluster(self, n_clusters=None):
        """Cluster using scaled RFM data."""
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(self.rfm_df)

        if n_clusters is None:
            inertia = []
            for i in range(1, 11):
                km = KMeans(n_clusters=i, n_init=10, random_state=42).fit(scaled_data)
                inertia.append(km.inertia_)
            n_clusters = KneeLocator(range(1, 11), inertia, curve="convex", direction="decreasing").knee

        self.model = KMeans(n_clusters=n_clusters, n_init=10, random_state=42).fit(scaled_data)
        self.rfm_df['Cluster'] = self.model.labels_

        # ADDED: Save the segmented data to CSV for marketing/sales teams
        self.rfm_df.to_csv('customer_segments.csv')
        print(f"Phase 2: Clustering complete (n_clusters={n_clusters}). Data saved to 'customer_segments.csv'.")

        return self.rfm_df

    # Visualization
    def phase_3_visualize(self):
        # 1. Elbow Plot
        plt.figure(figsize=(10, 4))
        # (Re-calc for plot)
        scaler = StandardScaler()
        scaled = scaler.fit_transform(self.rfm_df[['Recency', 'Frequency', 'Monetary']])
        inertias = [KMeans(n_clusters=i, n_init=10, random_state=42).fit(scaled).inertia_ for i in range(1, 11)]

        plt.subplot(1, 2, 1)
        plt.plot(range(1, 11), inertias, marker='o')
        plt.title('Elbow Method')
        plt.savefig('/content/elbow_plot.png')

        # 2. Scatter Plot
        plt.subplot(1, 2, 2)
        sns.scatterplot(data=self.rfm_df, x='Recency', y='Monetary', hue='Cluster', palette='viridis')
        plt.title('Clusters: Recency vs Monetary')
        plt.savefig('/content/cluster_plot.png')
        plt.show()

        # Add this line so the summary table is actually passed back
        summary = self.rfm_df.groupby('Cluster').mean()

        # THIS IS THE CRITICAL LINE:
        # Ensure the filename and path match the test exactly
        summary.to_csv('/content/segment_summary.csv')

        print("✅ Phase 3: Visualizations generated and saved to /content/")
        return summary

# --- EXECUTION BLOCK ---
file_name = "online_retail.csv"

# Initialize the pipeline
pipeline = RetailPipeline(f'/content/{file_name}')
pipeline.phase_0_prep()
pipeline.phase_1_rfm()
pipeline.phase_2_cluster()
summary = pipeline.phase_3_visualize()

# Print the final persona summary to console
print("\n--- Final Customer Persona Summary (Average Metrics) ---")
print(summary)

Writing week3_assignment.py


In [2]:
!pip install kneed

In [4]:
# This command tells Python to execute the script you just saved
!python week3_assignment.py

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='object')
✅ Phase 0: Data cleaning complete.
✅ Phase 1: RFM metrics calculated.
Phase 2: Clustering complete (n_clusters=4). Data saved to 'customer_segments.csv'.
Figure(1000x400)
✅ Phase 3: Visualizations generated and saved to /content/

--- Final Customer Persona Summary (Average Metrics) ---
            Recency  Frequency       Monetary
Cluster                                      
0        115.742690   1.383626     460.494193
1          5.000000  51.400000  101493.534000
2          6.758621  29.103448   16396.775586
3         29.645846   2.906895    1181.849725


In [5]:
import pandas as pd
df = pd.read_csv('/content/online_retail.csv', encoding='ISO-8859-1')
print(df.columns.tolist())

['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [ ]:
# This command tells Python to execute the script you just saved
!python week3_assignment.py

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='object')
✅ Phase 0: Data cleaning complete.
✅ Phase 1: RFM metrics calculated.
Phase 2: Clustering complete (n_clusters=4). Data saved to 'customer_segments.csv'.
Figure(1000x400)
✅ Phase 3: Visualizations generated and saved to /content/

--- Final Customer Persona Summary (Average Metrics) ---
            Recency   Frequency       Monetary
Cluster                                       
0        463.032032    2.212212     765.244446
1         67.005728    7.307732    3009.402451
2         25.942857  103.714286   83086.079771
3          3.500000  212.500000  436835.792500


In [6]:
%%writefile week3_assignment_Template.py

class RetailPipeline:
    def __init__(self, data_path):
        raise NotImplementedError("Not Yet Implemented")

    def phase_0_prep(self):
        raise NotImplementedError("Not Yet Implemented")

    def phase_1_rfm(self):
        raise NotImplementedError("Not Yet Implemented")

    def phase_2_cluster(self, n_clusters=None):
        raise NotImplementedError("Not Yet Implemented")

    def phase_3_visualize(self):
        raise NotImplementedError("Not Yet Implemented")

Writing week3_assignment_Template.py


In [7]:
%%writefile test.py

import unittest
import pandas as pd
import os

# Placeholder for the class injected by tester.py
RetailPipelineClass = None

class TestRetailPipeline(unittest.TestCase):
    def setUp(self):
        # The class is injected here
        self.pipeline = RetailPipelineClass('/content/online_retail.csv')

    # --- Phase 0: Prep Tests ---
    def test_0_prep_returns_df(self):
        df = self.pipeline.phase_0_prep()
        self.assertIsInstance(df, pd.DataFrame)

    def test_0_prep_snapshot_exists(self):
        self.pipeline.phase_0_prep()
        self.assertIsNotNone(self.pipeline.snapshot_date)

    def test_0_prep_drops_nans(self):
        df = self.pipeline.phase_0_prep()
        self.assertEqual(df['Customer ID'].isna().sum(), 0)

    def test_0_prep_no_cancelled(self):
        df = self.pipeline.phase_0_prep()
        self.assertFalse(df['Invoice'].astype(str).str.startswith('C').any())

    # --- Phase 1: RFM Tests ---
    def test_1_rfm_columns_exist(self):
        self.pipeline.phase_0_prep()
        df = self.pipeline.phase_1_rfm()
        self.assertIn('Recency', df.columns)
        self.assertIn('Frequency', df.columns)
        self.assertIn('Monetary', df.columns)

    def test_1_rfm_returns_df(self):
        self.pipeline.phase_0_prep()
        self.assertIsInstance(self.pipeline.phase_1_rfm(), pd.DataFrame)

    def test_1_rfm_non_empty(self):
        self.pipeline.phase_0_prep()
        self.assertFalse(self.pipeline.phase_1_rfm().empty)

    def test_1_rfm_index_is_customer(self):
        self.pipeline.phase_0_prep()
        df = self.pipeline.phase_1_rfm()
        self.assertIn('Customer ID', df.index.name or '') # Depending on your groupby result

    # --- Phase 2: Cluster Tests ---
    def test_2_cluster_assignment(self):
        self.pipeline.phase_0_prep()
        self.pipeline.phase_1_rfm()
        df = self.pipeline.phase_2_cluster()
        self.assertIn('Cluster', df.columns)

    def test_2_cluster_export(self):
        self.pipeline.phase_0_prep()
        self.pipeline.phase_1_rfm()
        self.pipeline.phase_2_cluster()
        self.assertTrue(os.path.exists('/content/customer_segments.csv'))

    def test_2_cluster_labels_exist(self):
        self.pipeline.phase_0_prep()
        self.pipeline.phase_1_rfm()
        self.pipeline.phase_2_cluster()
        self.assertIsNotNone(self.pipeline.model.labels_)

    def test_2_cluster_count(self):
        self.pipeline.phase_0_prep()
        self.pipeline.phase_1_rfm()
        df = self.pipeline.phase_2_cluster()
        self.assertGreaterEqual(df['Cluster'].nunique(), 1)

    # --- Phase 3: Visualization Tests ---
    def test_3_summary_export(self):
        self.pipeline.phase_0_prep()
        self.pipeline.phase_1_rfm()
        self.pipeline.phase_2_cluster()
        self.pipeline.phase_3_visualize()
        self.assertTrue(os.path.exists('/content/segment_summary.csv'))

    def test_3_summary_is_df(self):
        self.pipeline.phase_0_prep()
        self.pipeline.phase_1_rfm()
        self.pipeline.phase_2_cluster()
        summary = self.pipeline.phase_3_visualize()
        self.assertIsInstance(summary, pd.DataFrame)

    def test_3_summary_has_index(self):
        self.pipeline.phase_0_prep()
        self.pipeline.phase_1_rfm()
        self.pipeline.phase_2_cluster()
        summary = self.pipeline.phase_3_visualize()
        self.assertEqual(summary.index.name, 'Cluster')

    def test_3_summary_not_empty(self):
        self.pipeline.phase_0_prep()
        self.pipeline.phase_1_rfm()
        self.pipeline.phase_2_cluster()
        summary = self.pipeline.phase_3_visualize()
        self.assertFalse(summary.empty)

Writing test.py


In [8]:
%%writefile tester.py

import sys
import importlib
import unittest
import test

def run_tester(script_name):
    module_name = script_name.replace('.py', '')
    try:
        target_module = importlib.import_module(module_name)
        RetailPipeline = target_module.RetailPipeline
        test.RetailPipelineClass = RetailPipeline
    except Exception as e:
        print(f"Error loading {script_name}: {e}")
        return

    # 1. Audit Implementation
    not_implemented_count = 0
    try:
        # Check __init__ specifically
        RetailPipeline('/content/online_retail.csv')
    except NotImplementedError:
        print("Initialization is NOT implemented. Skipping tests.")
        not_implemented_count = 16 # All tests are effectively unimplemented

    # 2. Run Unit Tests ONLY if implemented
    if not_implemented_count == 0:
        print("\n--- Running Unit Tests ---")
        suite = unittest.TestLoader().loadTestsFromModule(test)
        runner = unittest.TextTestRunner(verbosity=1)
        result = runner.run(suite)
        passed = result.testsRun - len(result.failures) - len(result.errors)
        failed = len(result.failures) + len(result.errors)
    else:
        passed = 0
        failed = 0

    # 3. Report
    print("\n" + "="*30)
    print("      FINAL TEST REPORT      ")
    print("="*30)
    print(f"Passed          : {passed}")
    print(f"Failed          : {failed}")
    print(f"Not Implemented : {not_implemented_count}")
    print("="*30)

if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("Usage: python tester.py <filename.py>")
    else:
        run_tester(sys.argv[1])

Writing tester.py


In [ ]:
!python tester.py week3_assignment_Template.py

Initialization is NOT implemented. Skipping tests.

      FINAL TEST REPORT      
Passed          : 0
Failed          : 0
Not Implemented : 16


In [9]:
!python tester.py week3_assignment.py

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='object')
✅ Phase 0: Data cleaning complete.
✅ Phase 1: RFM metrics calculated.
Phase 2: Clustering complete (n_clusters=4). Data saved to 'customer_segments.csv'.
Figure(1000x400)
✅ Phase 3: Visualizations generated and saved to /content/

--- Final Customer Persona Summary (Average Metrics) ---
            Recency   Frequency       Monetary
Cluster                                       
0        396.826746    2.236213     779.208977
1         80.282928    7.049872    2914.836042
2          5.250000  174.500000  375086.750000
3         24.714286   92.742857   67030.335486
Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='object')

--- Running Unit Tests ---
Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtyp

In [ ]:
# 1. Zip all files in the "My Project Files" folder into a zip file
!zip -r /content/my_project_files.zip "/content/My Project Files"

# Trigger the download
from google.colab import files
files.download('/content/my_project_files.zip')

  adding: content/My Project Files/ (stored 0%)
  adding: content/My Project Files/online_retail.csv (deflated 84%)
  adding: content/My Project Files/elbow_plot.png (deflated 14%)
  adding: content/My Project Files/cluster_plot.png (deflated 7%)
  adding: content/My Project Files/week3_assignment.py (deflated 61%)
  adding: content/My Project Files/customer_segments.csv (deflated 63%)
  adding: content/My Project Files/segment_summary.csv (deflated 37%)
  adding: content/My Project Files/week3_assignment_Template.py (deflated 69%)
  adding: content/My Project Files/tester.py (deflated 58%)
  adding: content/My Project Files/test.py (deflated 79%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
# 1. Zip all files in the "My Project Files" folder into a zip file
!zip -r /content/my_project_files.zip "/content/My Project Files"
# Trigger the download
from google.colab import files
files.download('/content/my_project_files.zip')


  adding: content/My Project Files/ (stored 0%)
  adding: content/My Project Files/cluster_plot.png (deflated 7%)
  adding: content/My Project Files/segment_summary.csv (deflated 37%)
  adding: content/My Project Files/test.py (deflated 79%)
  adding: content/My Project Files/tester.py (deflated 58%)
  adding: content/My Project Files/week3_assignment.py (deflated 61%)
  adding: content/My Project Files/customer_segments.csv (deflated 63%)
  adding: content/My Project Files/week3_assignment_Template.py (deflated 69%)
  adding: content/My Project Files/elbow_plot.png (deflated 14%)
  adding: content/My Project Files/online_retail.csv (deflated 84%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>